# 05 — Regroupement des services Internet (après retrait de 4 variables)

**Point de départ :** `PhoneService`, `MultipleLines`, `gender` et `TotalCharges` ont déjà été retirées.
Il reste la zone à forte association de la matrice de V de Cramér : `InternetService` et les six services
(`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`),
avec des V de 0,71 à 0,77.

**Démarche :**

1. Regrouper les services en deux indices : `Security_Index` (0 à 4) et `Streaming_Pack` (0 à 2).
2. **Refaire les matrices d'association** (V de Cramér, rapport de corrélation, VIF) avant / après, pour voir
   si le phénomène disparaît.
3. Comparer plusieurs scénarios par **validation croisée sur l'entraînement** (AUC, rappel, précision, F1),
   puis confirmer **une seule fois** sur le test.

> Les services sont stockés sous forme de texte (« Yes », « No », « No internet service »). On les convertit
> donc en 0/1 avant de les additionner (une somme directe sur du texte ne fonctionnerait pas).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

# Permet d'importer le package src depuis le dossier notebooks/
sys.path.append(str(Path.cwd().parent))
from src.data.clean_data import load_data, clean_data

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Paramètres (garder cohérents avec les notebooks précédents et avec src/features/build_features.py)
TEST_SIZE = 0.2
RANDOM_STATE = 42
TOLERANCE_AUC = 0.005
TOLERANCE_RAPPEL = 0.01

# Variables déjà retirées du projet (ajouter ici d'autres si besoin)
VARIABLES_DEJA_RETIREES = ["PhoneService", "MultipleLines", "gender", "TotalCharges"]

# Variables numériques restantes
COLONNES_NUMERIQUES = [
    c for c in ["tenure", "MonthlyCharges", "TotalCharges"] if c not in VARIABLES_DEJA_RETIREES
]
SERVICES_SECURITE = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
SERVICES_STREAMING = ["StreamingTV", "StreamingMovies"]
SERVICES_INTERNET = SERVICES_SECURITE + SERVICES_STREAMING

## 1. Données et séparation entraînement / test

In [ ]:
df = clean_data(load_data())
X = df.drop(columns=["Churn"] + VARIABLES_DEJA_RETIREES, errors="ignore")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Entraînement : {X_train.shape} | Test : {X_test.shape}")
print("Variables :", list(X_train.columns))

## 2. Fonctions de transformation

Elles sont écrites pour être placées **dans le pipeline** (`FunctionTransformer`) : l'API continue de
recevoir les variables brutes, et c'est le pipeline qui calcule les indices. Ainsi, l'entraînement et
l'inférence appliquent exactement les mêmes transformations.

- `Security_Index` : nombre de services de sécurité souscrits (0 à 4) ;
- `Streaming_Pack` : nombre de services de streaming souscrits (0 à 2).

> **Attention :** un client sans Internet et un client avec Internet mais sans aucun service ont tous les deux
> un indice égal à 0. Cette distinction n'est pas perdue, car `InternetService` est conservée.

In [ ]:
def vers_binaire(serie):
    # "Yes" -> 1, tout le reste ("No", "No internet service") -> 0 ; accepte aussi des colonnes déjà numériques
    if pd.api.types.is_numeric_dtype(serie):
        return (serie > 0).astype(int)
    return (serie == "Yes").astype(int)


def transformer_variables(X, recoder, regroupements):
    # recoder : remplace "No internet service" par "No" dans les services restants
    # regroupements : dict {nom_nouvelle_variable: [colonnes]} -> nombre de services souscrits
    X = X.copy()
    if recoder:
        for col in SERVICES_INTERNET:
            if col in X.columns and not pd.api.types.is_numeric_dtype(X[col]):
                X[col] = X[col].replace("No internet service", "No")
    for nom, colonnes in regroupements.items():
        X[nom] = sum(vers_binaire(X[c]) for c in colonnes)
    return X


REGROUPEMENT_EXEMPLE = {"Security_Index": SERVICES_SECURITE, "Streaming_Pack": SERVICES_STREAMING}

# Vérification rapide sur quelques lignes
apercu = transformer_variables(X_train, recoder=True, regroupements=REGROUPEMENT_EXEMPLE)
apercu[SERVICES_INTERNET + ["Security_Index", "Streaming_Pack"]].head()

## 3. Matrices d'association : avant / après regroupement

### 3.1 V de Cramér entre variables catégorielles

« Avant » : variables d'origine (sans `PhoneService` ni `MultipleLines`).
« Après » : les six services sont remplacés par `Security_Index` et `Streaming_Pack`
(traités ici comme des catégories).

In [ ]:
def cramers_v(x, y, correction=True):
    table = pd.crosstab(x, y)
    if min(table.shape) < 2:
        return 0.0
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.values.sum()
    r, k = table.shape
    phi2 = chi2 / n
    if correction:
        phi2 = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
        r = r - ((r - 1) ** 2) / (n - 1)
        k = k - ((k - 1) ** 2) / (n - 1)
    denominateur = min(k - 1, r - 1)
    return float(np.sqrt(phi2 / denominateur)) if denominateur > 0 else 0.0


def matrice_cramers_v(data, colonnes):
    m = pd.DataFrame(np.eye(len(colonnes)), index=colonnes, columns=colonnes)
    for i, a in enumerate(colonnes):
        for b in colonnes[i + 1:]:
            m.loc[a, b] = m.loc[b, a] = cramers_v(data[a].astype(str), data[b].astype(str))
    return m


def afficher_matrice(m, titre):
    masque = np.triu(np.ones_like(m, dtype=bool), k=1)
    plt.figure(figsize=(12, 9))
    sns.heatmap(m, mask=masque, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
    plt.title(titre)
    plt.tight_layout()
    plt.show()


def paires_les_plus_fortes(m, n=8):
    cols = list(m.columns)
    paires = [
        {"variable_a": a, "variable_b": b, "cramers_v": m.loc[a, b]}
        for i, a in enumerate(cols) for b in cols[i + 1:]
    ]
    return pd.DataFrame(paires).sort_values("cramers_v", ascending=False).head(n).round(3)


categorielles_avant = [c for c in X_train.columns if c not in COLONNES_NUMERIQUES]
v_avant = matrice_cramers_v(X_train, categorielles_avant)
afficher_matrice(v_avant, "V de Cramér : AVANT regroupement")
display(paires_les_plus_fortes(v_avant))

In [ ]:
X_train_apres = transformer_variables(X_train, recoder=True, regroupements=REGROUPEMENT_EXEMPLE)
X_train_apres = X_train_apres.drop(columns=SERVICES_INTERNET)

categorielles_apres = [c for c in X_train_apres.columns if c not in COLONNES_NUMERIQUES]
v_apres = matrice_cramers_v(X_train_apres, categorielles_apres)
afficher_matrice(v_apres, "V de Cramér : APRÈS regroupement (Security_Index, Streaming_Pack)")
display(paires_les_plus_fortes(v_apres))

hors_diag = lambda m: m.values[np.triu_indices_from(m.values, k=1)]
print(f"V maximal avant : {hors_diag(v_avant).max():.2f} | après : {hors_diag(v_apres).max():.2f}")

### 3.2 Variables numériques et rapport de corrélation

Le V de Cramér ne couvre pas la relation entre une variable numérique et une catégorielle. Or
`MonthlyCharges` dépend directement des services souscrits. On mesure ce lien avec le **rapport de
corrélation η** (de 0 à 1), avant et après regroupement.

In [ ]:
def rapport_correlation(categorie, valeurs):
    moyenne = valeurs.mean()
    ss_total = ((valeurs - moyenne) ** 2).sum()
    ss_inter = sum(len(g) * (g.mean() - moyenne) ** 2 for _, g in valeurs.groupby(categorie.astype(str)))
    return float(np.sqrt(ss_inter / ss_total)) if ss_total > 0 else 0.0


def matrice_eta(data, numeriques, categorielles):
    return pd.DataFrame(
        {cat: [rapport_correlation(data[cat], data[num]) for num in numeriques] for cat in categorielles},
        index=numeriques,
    )


eta_avant = matrice_eta(X_train, COLONNES_NUMERIQUES, categorielles_avant)
eta_apres = matrice_eta(X_train_apres, COLONNES_NUMERIQUES, categorielles_apres)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, m, titre in zip(axes, [eta_avant, eta_apres], ["AVANT regroupement", "APRÈS regroupement"]):
    sns.heatmap(m, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"Rapport de corrélation η (numérique vs catégorielle) : {titre}")
plt.tight_layout()
plt.show()

In [ ]:
numeriques_apres = COLONNES_NUMERIQUES + ["Security_Index", "Streaming_Pack"]
corr = X_train_apres[numeriques_apres].corr(method="spearman")

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Corrélation de Spearman (variables numériques et indices)")
plt.tight_layout()
plt.show()

## 4. Les services sont-ils liés à `Churn` ?

Avant de remplacer les six services par des indices, on vérifie qu'on ne perd pas de signal : si certains
services sont nettement plus liés à `Churn` que d'autres, un indice global peut le diluer
(ce qui motive le scénario « hybride » plus bas).

In [ ]:
colonnes_v = SERVICES_INTERNET + ["InternetService", "Security_Index", "Streaming_Pack"]
base_churn = transformer_variables(X_train, recoder=True, regroupements=REGROUPEMENT_EXEMPLE)

v_churn = pd.Series({
    col: cramers_v(base_churn[col].astype(str), y_train.astype(str)) for col in colonnes_v
}).sort_values(ascending=False)

couleurs = ["darkorange" if c in ("Security_Index", "Streaming_Pack") else "steelblue" for c in v_churn.index]
plt.figure(figsize=(8, 5))
plt.barh(v_churn.index[::-1], v_churn.values[::-1], color=couleurs[::-1])
plt.xlabel("V de Cramér avec Churn")
plt.title("Lien avec Churn : services d'origine (bleu) et indices (orange)")
plt.tight_layout()
plt.show()

avec_internet = base_churn.assign(Churn=y_train)
avec_internet = avec_internet[avec_internet["InternetService"] != "No"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, indice in zip(axes, ["Security_Index", "Streaming_Pack"]):
    taux = avec_internet.groupby(indice)["Churn"].mean().mul(100)
    ax.bar(taux.index.astype(str), taux.values, color="indianred")
    ax.set_xlabel(f"{indice} (clients avec Internet)")
    ax.set_ylabel("Taux de churn (%)")
plt.tight_layout()
plt.show()

## 5. Scénarios

| Scénario | Principe |
|---|---|
| A — origine | les 6 services tels quels (avec « No internet service ») |
| B — recodage seul | « No internet service » remplacé par « No » ; les 6 services conservés |
| C — deux indices | `Security_Index` + `Streaming_Pack` à la place des 6 services (proposition de départ) |
| D — indice global | un seul indice `Services_Internet_Total` (0 à 6) à la place des 6 services |
| E — hybride | on garde `OnlineSecurity` et `TechSupport` ; les 4 autres sont regroupés dans `Autres_Services_Index` |

Pour chaque scénario, on mesure aussi le **VIF** (facteur d'inflation de la variance) : plus il est élevé,
plus il y a de multicolinéarité (au-delà de 5 à 10, c'est préoccupant).

In [ ]:
INDIVIDUELLES_HYBRIDE = ["OnlineSecurity", "TechSupport"]   # à ajuster d'après la section 4
AUTRES_SERVICES = [s for s in SERVICES_INTERNET if s not in INDIVIDUELLES_HYBRIDE]

SCENARIOS = {
    "A_origine": {"recoder": False, "regroupements": {}, "retirer": []},
    "B_recodage_seul": {"recoder": True, "regroupements": {}, "retirer": []},
    "C_deux_indices": {
        "recoder": True,
        "regroupements": REGROUPEMENT_EXEMPLE,
        "retirer": SERVICES_INTERNET,
    },
    "D_indice_global": {
        "recoder": True,
        "regroupements": {"Services_Internet_Total": SERVICES_INTERNET},
        "retirer": SERVICES_INTERNET,
    },
    "E_hybride": {
        "recoder": True,
        "regroupements": {"Autres_Services_Index": AUTRES_SERVICES},
        "retirer": AUTRES_SERVICES,
    },
}
REFERENCE = "A_origine"

MODELES = {
    "regression_logistique": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "xgboost": lambda: XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        eval_metric="logloss", random_state=RANDOM_STATE,
        # pondération de la classe minoritaire (équivalent de class_weight="balanced")
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    ),
}


def construire_regroupement(config):
    return FunctionTransformer(
        transformer_variables,
        kw_args={"recoder": config["recoder"], "regroupements": config["regroupements"]},
    )


def construire_preprocesseur(config, **options_onehot):
    numeriques = COLONNES_NUMERIQUES + list(config["regroupements"])
    categorielles = [
        c for c in X_train.columns
        if c not in COLONNES_NUMERIQUES and c not in config["retirer"]
    ]
    return ColumnTransformer([
        ("num", StandardScaler(), numeriques),
        ("cat", OneHotEncoder(handle_unknown="ignore", **options_onehot), categorielles),
    ])


def construire_pipeline(config, modele):
    return Pipeline([
        ("regroupement", construire_regroupement(config)),
        ("preprocessing", construire_preprocesseur(config)),
        ("modele", modele),
    ])


def calculer_vif(matrice, noms):
    vifs = []
    for j in range(matrice.shape[1]):
        autres = np.delete(matrice, j, axis=1)
        r2 = LinearRegression().fit(autres, matrice[:, j]).score(autres, matrice[:, j])
        vifs.append(np.inf if r2 > 1 - 1e-10 else 1 / (1 - r2))
    return pd.Series(vifs, index=noms)


n_colonnes, vif_par_scenario = {}, {}
for nom, config in SCENARIOS.items():
    pipe_vif = Pipeline([
        ("regroupement", construire_regroupement(config)),
        # drop="first" évite la colinéarité parfaite propre à l'encodage one-hot complet
        ("preprocessing", construire_preprocesseur(config, drop="first", sparse_output=False)),
    ]).fit(X_train)
    matrice = pipe_vif.transform(X_train)
    noms = pipe_vif.named_steps["preprocessing"].get_feature_names_out()
    n_colonnes[nom] = construire_pipeline(config, LogisticRegression())[:-1].fit(X_train, y_train).transform(X_train).shape[1]
    vif_par_scenario[nom] = calculer_vif(matrice, noms)

resume_vif = pd.DataFrame({
    "colonnes_apres_encodage": pd.Series(n_colonnes),
    "vif_max": pd.Series({n: v.max() for n, v in vif_par_scenario.items()}),
    "colonnes_vif_sup_5": pd.Series({n: int((v > 5).sum()) for n, v in vif_par_scenario.items()}),
})
display(resume_vif.round(1))

print("\nVIF les plus élevés, scénario A :")
display(vif_par_scenario[REFERENCE].sort_values(ascending=False).head(8).round(1))
print("VIF les plus élevés, scénario C :")
display(vif_par_scenario["C_deux_indices"].sort_values(ascending=False).head(8).round(1))

## 6. Validation croisée sur le jeu d'entraînement

5 plis répétés 3 fois, mêmes découpages pour tous les scénarios et tous les modèles.

In [ ]:
SCORING = {"auc": "roc_auc", "rappel": "recall", "precision": "precision", "f1": "f1"}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

lignes = []
for nom_modele, fabrique in MODELES.items():
    for nom, config in SCENARIOS.items():
        scores = cross_validate(
            construire_pipeline(config, fabrique()), X_train, y_train,
            cv=cv, scoring=SCORING, n_jobs=-1,
        )
        ligne = {
            "modele": nom_modele, "scenario": nom,
            "colonnes": n_colonnes[nom], "vif_max": vif_par_scenario[nom].max(),
        }
        for metrique in SCORING:
            ligne[f"{metrique}_moy"] = scores[f"test_{metrique}"].mean()
            ligne[f"{metrique}_std"] = scores[f"test_{metrique}"].std()
        lignes.append(ligne)

resultats_cv = pd.DataFrame(lignes).set_index(["modele", "scenario"])
display(resultats_cv[
    ["colonnes", "vif_max", "auc_moy", "auc_std", "rappel_moy", "precision_moy", "f1_moy"]
].round(4))

## 7. Décision

Un scénario est **équivalent** à la référence (pour un modèle donné) si son AUC et son rappel moyens
ne sont pas plus bas que ceux de A de plus que les tolérances (paramètres en haut du notebook). On retient
le scénario équivalent **pour les deux modèles** qui a le moins de colonnes (puis le VIF le plus bas).

In [ ]:
equivalents = {}
for nom_modele in MODELES:
    ref = resultats_cv.loc[(nom_modele, REFERENCE)]
    ok = [
        nom for nom in SCENARIOS
        if resultats_cv.loc[(nom_modele, nom), "auc_moy"] >= ref["auc_moy"] - TOLERANCE_AUC
        and resultats_cv.loc[(nom_modele, nom), "rappel_moy"] >= ref["rappel_moy"] - TOLERANCE_RAPPEL
    ]
    equivalents[nom_modele] = set(ok)
    print(f"{nom_modele} : équivalents à A -> {sorted(ok)}")

communs = set.intersection(*equivalents.values())
retenu = min(communs, key=lambda n: (n_colonnes[n], vif_par_scenario[n].max()))
print(f"\nScénario retenu : {retenu} ({n_colonnes[retenu]} colonnes, VIF max = {vif_par_scenario[retenu].max():.1f})")

tableau = resultats_cv["auc_moy"].unstack("modele")
ax = tableau.plot.barh(figsize=(10, 4))
ax.set_xlim(tableau.min().min() - 0.02, tableau.max().max() + 0.02)
ax.set_xlabel("AUC moyenne (validation croisée)")
ax.set_title("Comparaison des scénarios")
plt.tight_layout()
plt.show()

## 8. Confirmation sur le jeu de test

Une seule évaluation par scénario et par modèle. Le test ne sert pas à changer le choix de la section 7.

In [ ]:
lignes_test = []
for nom_modele, fabrique in MODELES.items():
    for nom, config in SCENARIOS.items():
        pipe = construire_pipeline(config, fabrique()).fit(X_train, y_train)
        proba = pipe.predict_proba(X_test)[:, 1]
        predictions = pipe.predict(X_test)
        lignes_test.append({
            "modele": nom_modele, "scenario": nom, "colonnes": n_colonnes[nom],
            "auc": roc_auc_score(y_test, proba),
            "rappel": recall_score(y_test, predictions),
            "precision": precision_score(y_test, predictions),
            "f1": f1_score(y_test, predictions),
        })

resultats_test = pd.DataFrame(lignes_test).set_index(["modele", "scenario"])
display(resultats_test.round(4))

## 9. Conclusion et mise en production

- Si le scénario retenu est **A**, on garde les services d'origine : la redondance ne gêne pas le modèle.
- Sinon, on reporte la transformation retenue **dans le pipeline du projet** (pas seulement ici) :
  1. déplacer `vers_binaire` et `transformer_variables` dans `src/features/` ;
  2. les placer en première étape du pipeline (`FunctionTransformer`) dans `src/features/build_features.py`,
     et adapter les listes de colonnes (numériques : les nouveaux indices ; catégorielles : sans les services retirés) ;
  3. l'API (`api/main.py`) continue de recevoir les variables brutes, donc le schéma Pydantic ne change pas
     (sauf pour les quatre variables déjà retirées) ;
  4. réentraîner, régénérer `preprocessor.joblib` / `model.joblib`, ajouter un test unitaire pour
     `transformer_variables`, puis lancer `pytest`.